# Random Semantic Algebra v2 — all F1 ideas
One-click Colab for the complete experiment suite. The core library is stored as three compressed text parts on the same branch and loaded below.

In [ ]:
!pip -q install datasets sentence-transformers scikit-learn pandas numpy pyarrow

In [ ]:
from pathlib import Path
import base64,gzip,urllib.request,shutil
import numpy as np, pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
SEED=7; FULL_RUN=True; STRICT_PROTOCOL=False
N_PRODUCTS=30_000 if FULL_RUN else 8_000
N_CONCEPTS=14 if FULL_RUN else 6
K_COORDS=28 if FULL_RUN else 12
CANDIDATE_POOL=128 if FULL_RUN else 48
PAIR_LUTS=4 if FULL_RUN else 2
WHITENING_GAMMAS=(.25,.5,.75,1.0) if FULL_RUN else (.5,)
OUT=Path('/content/random_semantic_algebra_v2_results'); OUT.mkdir(exist_ok=True)
base='https://raw.githubusercontent.com/hanialshater/LSH_Memory/refs/heads/chatgpt/random-semantic-algebra-v2/experiments/'
blob=''.join(urllib.request.urlopen(base+f'rsa_v2_lib.part{i}').read().decode() for i in range(3))
exec(gzip.decompress(base64.b64decode(blob)).decode())
print('RSA v2 library loaded')

In [ ]:
rng=np.random.default_rng(SEED)
keep=['id','gender','masterCategory','subCategory','articleType','baseColour','season','usage','productDisplayName']
ds=load_dataset('ashraq/fashion-product-images-small',split='train')
ds=ds.remove_columns([c for c in ds.column_names if c not in keep])
if len(ds)>N_PRODUCTS: ds=ds.select(rng.choice(len(ds),size=N_PRODUCTS,replace=False).tolist())
df=ds.to_pandas()
for c in keep:
    if c!='id': df[c]=df[c].fillna('Unknown').astype(str)
enc=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X=enc.encode(df.productDisplayName.tolist(),batch_size=256,show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype(np.float32)
facets=['baseColour','articleType','subCategory','usage','gender','masterCategory']
cand=[]
for col in facets:
    for value,count in df[col].value_counts().items():
        p=count/len(df)
        if count>=350 and .025<=p<=.70: cand.append((col,value,int(count),float(p)))
picked=[]; per={c:0 for c in facets}
for item in sorted(cand,key=lambda z:z[2],reverse=True):
    if per[item[0]]<4: picked.append(item); per[item[0]]+=1
    if len(picked)>=N_CONCEPTS: break
concepts=[(c,v) for c,v,_,_ in picked]
Y=np.column_stack([(df[c].to_numpy()==v) for c,v in concepts]).astype(bool)
display(pd.DataFrame(picked,columns=['field','value','count','prevalence']))

In [ ]:
sp=make_protocol_split(len(df),SEED,STRICT_PROTOCOL)
X_fit,X_cal,X_test=X[sp.fit_idx],X[sp.cal_idx],X[sp.test_idx]
Y_fit,Y_cal,Y_test=Y[sp.fit_idx],Y[sp.cal_idx],Y[sp.test_idx]
substrates,runs,board=run_all_novel_ideas(X_fit=X_fit,X_cal=X_cal,X_test=X_test,Y_fit=Y_fit,Y_cal=Y_cal,Y_test=Y_test,concepts=concepts,seed=SEED,k=K_COORDS,candidate_pool=CANDIDATE_POOL,n_pair_luts=PAIR_LUTS,whitening_gammas=WHITENING_GAMMAS)
display(board)

In [ ]:
def cal_f1(r):
    return float(np.mean([f1_score(Y_cal[:,c],r.scores_cal[:,c]>=best_f1_threshold(r.scores_cal[:,c],Y_cal[:,c]),zero_division=0) for c in range(Y_cal.shape[1])]))
cal_board=pd.DataFrame([{'run':k,'cal_mean_f1':cal_f1(r),'test_mean_f1':r.mean_f1,'test_mean_ap':r.mean_ap,'sparse':r.sparse} for k,r in runs.items()]).sort_values('cal_mean_f1',ascending=False)
best_key=cal_board[cal_board['sparse']].iloc[0]['run']; best=runs[best_key]
print('best sparse by calibration:',best_key); display(cal_board)
comp=evaluate_composition_algebras(best,Y_cal,Y_test,concepts)
print('global soft-min tau:',comp['best_tau']); display(comp['pair_summary']); display(comp['triple_summary'])

In [ ]:
geom=pd.DataFrame([{'substrate':n,'bits':s.bits,'m':s.meta['m'],'bytes/item':s.item_bytes_theoretical,'geometry_corr':geometry_correlation(X_test,s,SEED)} for n,s in substrates.items()]).sort_values('geometry_corr',ascending=False)
display(geom)
save_run_artifacts(OUT,runs,board,comp); geom.to_csv(OUT/'geometry.csv',index=False); cal_board.to_csv(OUT/'calibration_leaderboard.csv',index=False)
print('results zip:',shutil.make_archive('/content/random_semantic_algebra_v2_results','zip',OUT))

Set `STRICT_PROTOCOL=True` before using the numbers in a paper. The readable, uncompressed notebook is also attached in ChatGPT.